# Understanding Confidence Intervals: An Interactive Journey

Welcome! In this notebook, we'll explore one of the most important concepts in statistics: **confidence intervals**. By the end, you'll understand not just how to calculate them, but what they really mean and why they matter.

## What You'll Learn

- 🎯 What confidence intervals are and why we need them
- 📊 The intuition behind confidence levels (90%, 95%, 99%)
- 🔍 How sample size and variability affect interval width
- ✅ The correct interpretation (and common misconceptions)
- 🛠️ How to calculate confidence intervals from scratch
- 🎨 How to visualize and interpret them in real scenarios

Let's begin!

In [ ]:
# Setup: Import all necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd
from IPython.display import display, Markdown

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib for better-looking plots
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("✓ Libraries loaded successfully!")
print("✓ Random seed set to 42 for reproducibility")

---
## Part 1: The Problem - Why We Need Confidence Intervals

Imagine you're a data scientist at a company launching a new product. You want to know the **average customer satisfaction score** for millions of potential customers, but you can only survey a small sample. 

### The Challenge

- 🌍 **Population**: All potential customers (millions)
- 👥 **Sample**: Only 100 customers surveyed
- ❓ **Question**: What can we say about the true population mean based on our sample?

Let's simulate this scenario!

In [ ]:
# Let's create a "population" of customer satisfaction scores
# In reality, we'd never know this - but for learning, we'll peek!
TRUE_POPULATION_MEAN = 7.5  # On a scale of 1-10
TRUE_POPULATION_STD = 1.8

# Generate a large population
population_size = 1_000_000
population = np.random.normal(TRUE_POPULATION_MEAN, TRUE_POPULATION_STD, population_size)

# Keep scores between 1 and 10
population = np.clip(population, 1, 10)

print(f"🌍 Population Statistics:")
print(f"   True Mean: {TRUE_POPULATION_MEAN:.2f}")
print(f"   True Std Dev: {TRUE_POPULATION_STD:.2f}")
print(f"   Population Size: {population_size:,}")

# Now let's take just one sample (what we'd actually have)
sample_size = 100
sample = np.random.choice(population, size=sample_size, replace=False)

print(f"\n👥 Sample Statistics:")
print(f"   Sample Mean: {sample.mean():.2f}")
print(f"   Sample Std Dev: {sample.std(ddof=1):.2f}")
print(f"   Sample Size: {sample_size}")

print(f"\n❓ Difference from truth: {abs(sample.mean() - TRUE_POPULATION_MEAN):.2f}")

### Key Observation

Notice that our sample mean is **close** to the true population mean, but not exactly the same! This is called **sampling error** - the natural variation that occurs when we sample.

**The Big Question**: How do we communicate our uncertainty about the true population mean based on our sample?

---
## Part 2: Point Estimates vs Interval Estimates

### Point Estimate
A **point estimate** is a single number that represents our best guess:
- Sample mean = 7.32 (for example)
- ❌ Problem: Gives no sense of uncertainty!

### Interval Estimate
An **interval estimate** is a range of plausible values:
- Between 7.0 and 7.6 (for example)  
- ✅ Better: Communicates uncertainty!

Let's visualize this difference:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Point Estimate
axes[0].axvline(sample.mean(), color='red', linewidth=3, label='Point Estimate')
axes[0].axvline(TRUE_POPULATION_MEAN, color='green', linewidth=2, linestyle='--', label='True Mean')
axes[0].set_xlim(6, 9)
axes[0].set_ylim(0, 1)
axes[0].set_xlabel('Satisfaction Score')
axes[0].set_title('Point Estimate: A Single Number')
axes[0].legend()
axes[0].set_yticks([])
axes[0].text(sample.mean(), 0.5, f'{sample.mean():.2f}', ha='center', fontsize=14, fontweight='bold')

# Plot 2: Interval Estimate (we'll calculate this properly later)
sample_mean = sample.mean()
margin = 0.35  # Simplified for now
lower = sample_mean - margin
upper = sample_mean + margin

axes[1].axvspan(lower, upper, alpha=0.3, color='blue', label='Interval Estimate')
axes[1].axvline(sample.mean(), color='red', linewidth=2, label='Sample Mean')
axes[1].axvline(TRUE_POPULATION_MEAN, color='green', linewidth=2, linestyle='--', label='True Mean')
axes[1].set_xlim(6, 9)
axes[1].set_ylim(0, 1)
axes[1].set_xlabel('Satisfaction Score')
axes[1].set_title('Interval Estimate: A Range of Plausible Values')
axes[1].legend()
axes[1].set_yticks([])
axes[1].text((lower + upper) / 2, 0.5, f'[{lower:.2f}, {upper:.2f}]', ha='center', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ The interval estimate captures our uncertainty!")

### 🤔 Reflection Question

Which approach gives more honest information: the point estimate or the interval estimate? Why?

---
## Part 3: The Sampling Distribution - The Foundation

To understand confidence intervals, we need to understand **sampling distributions**.

### The Concept
If we took many different samples and calculated the mean of each, those sample means would form their own distribution - the **sampling distribution of the mean**.

Let's see this in action!

In [ ]:
# Take 1000 different samples and calculate their means
n_simulations = 1000
sample_size = 100
sample_means = []

for i in range(n_simulations):
    sample = np.random.choice(population, size=sample_size, replace=False)
    sample_means.append(sample.mean())

sample_means = np.array(sample_means)

print(f"📊 Simulation Results ({n_simulations} samples of size {sample_size}):")
print(f"   Mean of sample means: {sample_means.mean():.3f}")
print(f"   True population mean: {TRUE_POPULATION_MEAN:.3f}")
print(f"   Std dev of sample means: {sample_means.std():.3f}")
print(f"   (This is called the 'Standard Error')")

Plot a histogram to understand the distribution.

In [ ]:
# Visualize the sampling distribution
plt.figure(figsize=(12, 6))
plt.hist(sample_means, bins=50, density=True, alpha=0.7, color='skyblue', edgecolor='black')
plt.axvline(TRUE_POPULATION_MEAN, color='green', linewidth=3, linestyle='--', label=f'True Mean ({TRUE_POPULATION_MEAN})')
plt.axvline(sample_means.mean(), color='red', linewidth=2, label=f'Mean of Sample Means ({sample_means.mean():.2f})')

# Add a normal curve overlay
x = np.linspace(sample_means.min(), sample_means.max(), 100)
plt.plot(x, stats.norm.pdf(x, sample_means.mean(), sample_means.std()), 
         'r-', linewidth=2, label='Normal Distribution Fit')

plt.xlabel('Sample Mean')
plt.ylabel('Density')
plt.title(f'Sampling Distribution of the Mean\n({n_simulations} samples of size {sample_size})')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("\n🎯 Key Insights:")
print("   1. Sample means cluster around the true population mean")
print("   2. The distribution looks approximately normal (Central Limit Theorem!)")
print("   3. Most sample means fall within a predictable range")

### 🔬 The Central Limit Theorem (Simplified)

The **Central Limit Theorem** states that when you take many samples and compute their means, those means will be approximately normally distributed, regardless of the population's original distribution!

**Why This Matters**: Because the sampling distribution is normal, we can use properties of the normal distribution to construct confidence intervals.

### 🧪 Your Turn: Experiment with Sample Size

Try changing the `sample_size` below and observe how it affects the sampling distribution:

In [ ]:
# Experiment: Compare different sample sizes
sample_sizes = [10, 50, 200]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, n in enumerate(sample_sizes):
    # Generate sampling distribution
    means = [np.random.choice(population, size=n, replace=False).mean() for _ in range(1000)]
    
    axes[idx].hist(means, bins=30, density=True, alpha=0.7, color='skyblue', edgecolor='black')
    axes[idx].axvline(TRUE_POPULATION_MEAN, color='green', linewidth=2, linestyle='--')
    axes[idx].set_xlabel('Sample Mean')
    axes[idx].set_ylabel('Density')
    axes[idx].set_title(f'Sample Size n={n}\nStd Error={np.std(means):.3f}')
    axes[idx].set_xlim(6.5, 8.5)

plt.tight_layout()
plt.show()

print("\n💡 Notice: As sample size increases, the sampling distribution becomes:")
print("   - Narrower (less variability)")
print("   - More concentrated around the true mean")
print("   - This means larger samples give more precise estimates!")

---
## Part 4: Introducing Confidence Intervals

Now we're ready for the main concept!

### What is a Confidence Interval?

A **confidence interval** is a range of values, calculated from sample data, that is likely to contain the true population parameter.

### The Key Idea

Since we know:
1. Sample means follow a normal distribution
2. Most sample means fall within a certain range of the true mean

We can **flip this logic**: Given a sample mean, we can construct a range that likely contains the true population mean!

Let's visualize this:

In [ ]:
# Generate 20 samples and their confidence intervals
np.random.seed(42)
n_samples = 20
sample_size = 100
confidence_level = 0.95

# For simplicity, we'll use a z-interval (we'll explain this later)
z_critical = stats.norm.ppf((1 + confidence_level) / 2)

intervals = []
captures = []

for i in range(n_samples):
    # Take a sample
    sample = np.random.choice(population, size=sample_size, replace=False)
    sample_mean = sample.mean()
    sample_std = sample.std(ddof=1)
    
    # Calculate confidence interval
    standard_error = sample_std / np.sqrt(sample_size)
    margin_of_error = z_critical * standard_error
    
    lower = sample_mean - margin_of_error
    upper = sample_mean + margin_of_error
    
    intervals.append((lower, upper, sample_mean))
    captures.append(lower <= TRUE_POPULATION_MEAN <= upper)

# Visualize
plt.figure(figsize=(12, 8))

for i, ((lower, upper, mean), captured) in enumerate(zip(intervals, captures)):
    color = 'green' if captured else 'red'
    plt.plot([lower, upper], [i, i], color=color, linewidth=2, alpha=0.7)
    plt.scatter(mean, i, color=color, s=50, zorder=3)

plt.axvline(TRUE_POPULATION_MEAN, color='blue', linewidth=3, linestyle='--', label=f'True Mean ({TRUE_POPULATION_MEAN})')
plt.xlabel('Satisfaction Score')
plt.ylabel('Sample Number')
plt.title(f'Twenty 95% Confidence Intervals\n{sum(captures)} out of {n_samples} captured the true mean ({sum(captures)/n_samples*100:.0f}%)')
plt.legend()
plt.grid(alpha=0.3)

# Add color legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='green', alpha=0.7, label='Captures true mean'),
                   Patch(facecolor='red', alpha=0.7, label='Misses true mean')]
plt.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

print(f"\n✅ {sum(captures)} intervals captured the true mean")
print(f"❌ {n_samples - sum(captures)} intervals missed the true mean")
print(f"\n📊 Capture rate: {sum(captures)/n_samples*100:.1f}% (expected: 95%)")

### 🎯 Critical Insight

Notice that:
- Most intervals (green) capture the true mean
- Some intervals (red) miss it - that's OK!
- The **intervals vary**, not the true mean (which is fixed)

This is the essence of confidence intervals!

---
## Part 5: Understanding Confidence Levels

### What Does "95% Confidence" Really Mean?

**Correct Interpretation**: 
If we repeated our sampling procedure many times and calculated a 95% confidence interval each time, approximately 95% of those intervals would contain the true population parameter.

**Common Misconceptions** (WRONG!):
- ❌ "There's a 95% probability the true mean is in this specific interval"
- ❌ "95% of the data falls in this interval"
- ❌ "We're 95% sure the true mean is in this interval"

**Why These Are Wrong**:
- The true mean is fixed (not random) - it either is or isn't in the interval
- The interval is what's random (it varies from sample to sample)
- The confidence is in the **method**, not in any particular interval

Let's demonstrate this with a simulation:

In [ ]:
# Simulate many confidence intervals to verify the confidence level
def simulate_confidence_intervals(n_simulations=1000, sample_size=100, confidence_level=0.95):
    z_critical = stats.norm.ppf((1 + confidence_level) / 2)
    captures = 0
    
    for _ in range(n_simulations):
        sample = np.random.choice(population, size=sample_size, replace=False)
        sample_mean = sample.mean()
        sample_std = sample.std(ddof=1)
        
        standard_error = sample_std / np.sqrt(sample_size)
        margin_of_error = z_critical * standard_error
        
        lower = sample_mean - margin_of_error
        upper = sample_mean + margin_of_error
        
        if lower <= TRUE_POPULATION_MEAN <= upper:
            captures += 1
    
    return captures / n_simulations

# Test different confidence levels
confidence_levels = [0.90, 0.95, 0.99]
n_sims = 1000

print(f"🔬 Simulation with {n_sims} samples each:\n")
print(f"{'Confidence Level':<20} {'Expected':<12} {'Observed':<12} {'Difference':<12}")
print("-" * 56)

results = []
for conf_level in confidence_levels:
    observed = simulate_confidence_intervals(n_sims, confidence_level=conf_level)
    results.append(observed)
    diff = observed - conf_level
    print(f"{conf_level*100:.0f}%{'':<17} {conf_level*100:.1f}%{'':<7} {observed*100:.1f}%{'':<7} {diff*100:+.1f}%")

print("\n✓ Observed rates closely match expected rates!")
print("  This confirms our confidence interval method is working correctly.")

Visualize the data distribution with a scatter plot.

In [ ]:
# Visualize the relationship between confidence level and interval width
sample = np.random.choice(population, size=100, replace=False)
sample_mean = sample.mean()
sample_std = sample.std(ddof=1)
standard_error = sample_std / np.sqrt(len(sample))

confidence_levels = [0.80, 0.90, 0.95, 0.99]
colors = ['lightblue', 'skyblue', 'steelblue', 'darkblue']

plt.figure(figsize=(12, 6))

for i, (conf_level, color) in enumerate(zip(confidence_levels, colors)):
    z = stats.norm.ppf((1 + conf_level) / 2)
    margin = z * standard_error
    lower = sample_mean - margin
    upper = sample_mean + margin
    
    y_pos = i
    plt.plot([lower, upper], [y_pos, y_pos], linewidth=8, color=color, alpha=0.7)
    plt.scatter(sample_mean, y_pos, color='red', s=100, zorder=3)
    plt.text(upper + 0.05, y_pos, f'{conf_level*100:.0f}% CI: [{lower:.2f}, {upper:.2f}]', 
             va='center', fontsize=11)

plt.axvline(TRUE_POPULATION_MEAN, color='green', linewidth=2, linestyle='--', label=f'True Mean ({TRUE_POPULATION_MEAN})')
plt.yticks(range(len(confidence_levels)), [f'{c*100:.0f}%' for c in confidence_levels])
plt.ylabel('Confidence Level')
plt.xlabel('Satisfaction Score')
plt.title('Effect of Confidence Level on Interval Width\nSame sample, different confidence levels')
plt.legend()
plt.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\n📏 Key Observation:")
print("   Higher confidence → Wider interval")
print("   Lower confidence → Narrower interval")
print("\n💡 Trade-off: Confidence vs Precision")

### 🤔 Reflection Questions

1. Why does higher confidence require a wider interval?
2. If you had to choose between a narrow interval (more precise) and high confidence, which would you choose and why?
3. In your own words, what does "95% confidence" mean?

---
## Part 6: The Mathematics - Building the Formula

Now let's understand the actual formula for a confidence interval:

### The General Formula

$$\text{Confidence Interval} = \text{Point Estimate} \pm (\text{Critical Value} \times \text{Standard Error})$$

Let's break down each component:

#### 1. Point Estimate
Usually the sample mean ($\bar{x}$)

#### 2. Standard Error (SE)
Measures the variability of the sample mean:

$$SE = \frac{s}{\sqrt{n}}$$

where:
- $s$ = sample standard deviation
- $n$ = sample size

#### 3. Critical Value
Depends on:
- The desired confidence level (90%, 95%, 99%, etc.)
- Whether we know the population std dev (Z) or not (t)

Let's calculate each component step-by-step:

In [ ]:
# Take a fresh sample for our calculation
np.random.seed(123)
sample = np.random.choice(population, size=100, replace=False)

print("📊 Our Sample Data:")
print(f"   First 10 values: {sample[:10].round(2)}")
print(f"   Sample size (n): {len(sample)}")

# Step 1: Calculate point estimate
point_estimate = sample.mean()
print(f"\n1️⃣ Point Estimate (sample mean):")
print(f"   x̄ = {point_estimate:.4f}")

# Step 2: Calculate standard error
sample_std = sample.std(ddof=1)  # ddof=1 for sample std dev
n = len(sample)
standard_error = sample_std / np.sqrt(n)

print(f"\n2️⃣ Standard Error:")
print(f"   s = {sample_std:.4f} (sample std dev)")
print(f"   n = {n} (sample size)")
print(f"   SE = s/√n = {sample_std:.4f}/√{n} = {standard_error:.4f}")

# Step 3: Find critical value for 95% confidence
confidence_level = 0.95
alpha = 1 - confidence_level
z_critical = stats.norm.ppf(1 - alpha/2)

print(f"\n3️⃣ Critical Value (z-score for 95% confidence):")
print(f"   Confidence level = {confidence_level*100:.0f}%")
print(f"   α (alpha) = {alpha:.2f}")
print(f"   α/2 = {alpha/2:.3f} (two-tailed)")
print(f"   z* = {z_critical:.4f}")
print(f"\n   💡 This means: {confidence_level*100:.0f}% of the normal distribution")
print(f"      falls within ±{z_critical:.2f} standard deviations of the mean")

Visualize the results.

In [ ]:
# Visualize where the critical value comes from
x = np.linspace(-4, 4, 1000)
y = stats.norm.pdf(x)

plt.figure(figsize=(12, 6))
plt.plot(x, y, 'b-', linewidth=2, label='Standard Normal Distribution')

# Shade the middle 95%
x_fill = x[(x >= -z_critical) & (x <= z_critical)]
y_fill = stats.norm.pdf(x_fill)
plt.fill_between(x_fill, y_fill, alpha=0.3, color='green', label=f'{confidence_level*100:.0f}% of distribution')

# Mark critical values
plt.axvline(-z_critical, color='red', linestyle='--', linewidth=2, label=f'Critical values: ±{z_critical:.2f}')
plt.axvline(z_critical, color='red', linestyle='--', linewidth=2)

# Shade the tails (2.5% each)
x_left = x[x <= -z_critical]
y_left = stats.norm.pdf(x_left)
plt.fill_between(x_left, y_left, alpha=0.3, color='red', label=f'{alpha/2*100:.1f}% in each tail')

x_right = x[x >= z_critical]
y_right = stats.norm.pdf(x_right)
plt.fill_between(x_right, y_right, alpha=0.3, color='red')

plt.xlabel('Standard Deviations from Mean (z-score)')
plt.ylabel('Probability Density')
plt.title(f'Critical Value for {confidence_level*100:.0f}% Confidence Level')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n📖 Interpretation:")
print(f"   For a 95% confidence level, we need to capture the middle 95% of the distribution.")
print(f"   This leaves {alpha/2*100:.1f}% in each tail.")
print(f"   The z-score that leaves {alpha/2*100:.1f}% in the upper tail is {z_critical:.4f}.")

Display the output.

In [ ]:
# Step 4: Calculate margin of error
margin_of_error = z_critical * standard_error

print("4️⃣ Margin of Error:")
print(f"   ME = z* × SE")
print(f"   ME = {z_critical:.4f} × {standard_error:.4f}")
print(f"   ME = {margin_of_error:.4f}")

# Step 5: Calculate confidence interval
lower_bound = point_estimate - margin_of_error
upper_bound = point_estimate + margin_of_error

print(f"\n5️⃣ Confidence Interval:")
print(f"   CI = x̄ ± ME")
print(f"   CI = {point_estimate:.4f} ± {margin_of_error:.4f}")
print(f"   CI = [{lower_bound:.4f}, {upper_bound:.4f}]")

print(f"\n" + "="*60)
print(f"🎯 FINAL RESULT:")
print(f"   We are 95% confident that the true population mean")
print(f"   is between {lower_bound:.2f} and {upper_bound:.2f}")
print(f"\n   (True mean: {TRUE_POPULATION_MEAN:.2f} " + 
      ("✓ Captured!" if lower_bound <= TRUE_POPULATION_MEAN <= upper_bound else "✗ Missed") + ")")
print("="*60)

### 🧪 Your Turn: Calculate a Confidence Interval

Try calculating a confidence interval with different parameters:

In [ ]:
# Your turn! Modify these parameters:
YOUR_CONFIDENCE_LEVEL = 0.90  # Try 0.80, 0.90, 0.95, 0.99

# Calculate z-critical value
z = stats.norm.ppf((1 + YOUR_CONFIDENCE_LEVEL) / 2)

# Calculate CI
margin = z * standard_error
ci_lower = point_estimate - margin
ci_upper = point_estimate + margin

print(f"\n{YOUR_CONFIDENCE_LEVEL*100:.0f}% Confidence Interval:")
print(f"   Critical value (z*) = {z:.4f}")
print(f"   Margin of error = {margin:.4f}")
print(f"   Confidence interval = [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"   Width = {ci_upper - ci_lower:.4f}")

# Compare with 95% CI
ci_95_width = 2 * (1.96 * standard_error)
print(f"\n   Comparison: 95% CI width = {ci_95_width:.4f}")
print(f"   Your CI is {((ci_upper - ci_lower) / ci_95_width - 1) * 100:+.1f}% wider/narrower")

---
## Part 7: Z-Intervals vs T-Intervals

So far, we've used **z-intervals** (based on the normal distribution). But in practice, we often use **t-intervals** instead. Why?

### When to Use Each

| Condition | Use | Distribution |
|-----------|-----|-------------|
| Population σ known & large sample | Z-interval | Normal (Z) |
| Population σ unknown & large sample (n > 30) | Either (nearly identical) | Normal or t |
| Population σ unknown & small sample (n ≤ 30) | T-interval | Student's t |

**In practice**: Since we rarely know the true population standard deviation, we almost always use **t-intervals**.

### The T-Distribution

The **t-distribution**:
- Looks similar to the normal distribution
- Has heavier tails (more spread out)
- Depends on **degrees of freedom** (df = n - 1)
- Approaches the normal distribution as n increases

Let's visualize the difference:

In [ ]:
# Compare t-distribution with normal distribution for different sample sizes
x = np.linspace(-4, 4, 1000)
sample_sizes = [5, 10, 30, 100]

plt.figure(figsize=(14, 8))

# Plot normal distribution
plt.plot(x, stats.norm.pdf(x), 'k-', linewidth=3, label='Normal (Z)', alpha=0.7)

# Plot t-distributions for different sample sizes
colors = ['red', 'orange', 'green', 'blue']
for n, color in zip(sample_sizes, colors):
    df = n - 1
    plt.plot(x, stats.t.pdf(x, df), color=color, linewidth=2, 
             label=f't-distribution (n={n}, df={df})', alpha=0.7)

plt.xlabel('Value')
plt.ylabel('Probability Density')
plt.title('Comparison: Normal vs t-Distributions\nNotice how t approaches normal as sample size increases')
plt.legend()
plt.grid(alpha=0.3)
plt.xlim(-4, 4)
plt.tight_layout()
plt.show()

print("\n📊 Key Observations:")
print("   • Small samples (n=5): t-distribution has much heavier tails")
print("   • Medium samples (n=30): t-distribution closer to normal")
print("   • Large samples (n=100): t-distribution nearly identical to normal")
print("\n💡 Heavier tails → Larger critical values → Wider confidence intervals")
print("   This accounts for extra uncertainty when estimating σ from small samples")

Display the output.

In [ ]:
# Compare critical values
confidence_level = 0.95
z_critical = stats.norm.ppf((1 + confidence_level) / 2)

print(f"\n Critical Values for {confidence_level*100:.0f}% Confidence:\n")
print(f"{'Sample Size':<15} {'df':<10} {'t*':<10} {'z*':<10} {'Difference':<12}")
print("-" * 57)

for n in [5, 10, 20, 30, 50, 100]:
    df = n - 1
    t_critical = stats.t.ppf((1 + confidence_level) / 2, df)
    diff = t_critical - z_critical
    print(f"{n:<15} {df:<10} {t_critical:<10.4f} {z_critical:<10.4f} {diff:+.4f}")

print("\n✓ Notice: As sample size increases, t* approaches z*")

### Computing a T-Interval

The formula is the same, but we use a t-critical value:

$$CI = \bar{x} \pm t_{\alpha/2, df} \times \frac{s}{\sqrt{n}}$$

Let's calculate both and compare:

In [ ]:
# Calculate both Z and T intervals for our sample
confidence_level = 0.95
sample_mean = sample.mean()
sample_std = sample.std(ddof=1)
n = len(sample)
se = sample_std / np.sqrt(n)

# Z-interval
z_crit = stats.norm.ppf((1 + confidence_level) / 2)
z_margin = z_crit * se
z_ci = (sample_mean - z_margin, sample_mean + z_margin)

# T-interval
t_crit = stats.t.ppf((1 + confidence_level) / 2, df=n-1)
t_margin = t_crit * se
t_ci = (sample_mean - t_margin, sample_mean + t_margin)

print(f"\n95% Confidence Intervals (n={n}):\n")
print(f"{'Type':<12} {'Critical Value':<18} {'Margin':<12} {'Interval':<25}")
print("-" * 67)
print(f"{'Z-interval':<12} {z_crit:<18.4f} {z_margin:<12.4f} [{z_ci[0]:.4f}, {z_ci[1]:.4f}]")
print(f"{'T-interval':<12} {t_crit:<18.4f} {t_margin:<12.4f} [{t_ci[0]:.4f}, {t_ci[1]:.4f}]")
print(f"\n{'Difference':<12} {t_crit - z_crit:<18.4f} {t_margin - z_margin:<12.4f} Width: {(t_ci[1]-t_ci[0]) - (z_ci[1]-z_ci[0]):.4f}")

# Visualize
plt.figure(figsize=(12, 5))
plt.axvspan(z_ci[0], z_ci[1], alpha=0.3, color='blue', label='Z-interval')
plt.axvspan(t_ci[0], t_ci[1], alpha=0.3, color='red', label='T-interval')
plt.axvline(sample_mean, color='black', linewidth=2, label='Sample mean')
plt.axvline(TRUE_POPULATION_MEAN, color='green', linewidth=2, linestyle='--', label='True mean')
plt.xlabel('Satisfaction Score')
plt.title(f'Comparison: Z-interval vs T-interval (n={n})')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n💡 With n={n}, the intervals are very similar.")
print(f"   For smaller samples, the t-interval would be noticeably wider.")

### 🧪 Your Turn: T-Interval with Small Sample

Let's see the difference with a small sample:

In [ ]:
# Take a small sample and compare intervals
small_sample = np.random.choice(population, size=10, replace=False)
small_n = len(small_sample)
small_mean = small_sample.mean()
small_std = small_sample.std(ddof=1)
small_se = small_std / np.sqrt(small_n)

# Calculate intervals
z_crit = stats.norm.ppf((1 + 0.95) / 2)
t_crit = stats.t.ppf((1 + 0.95) / 2, df=small_n-1)

z_ci_small = (small_mean - z_crit * small_se, small_mean + z_crit * small_se)
t_ci_small = (small_mean - t_crit * small_se, small_mean + t_crit * small_se)

print(f"\n95% Confidence Intervals (n={small_n}):\n")
print(f"Z-interval: [{z_ci_small[0]:.4f}, {z_ci_small[1]:.4f}]  (width: {z_ci_small[1]-z_ci_small[0]:.4f})")
print(f"T-interval: [{t_ci_small[0]:.4f}, {t_ci_small[1]:.4f}]  (width: {t_ci_small[1]-t_ci_small[0]:.4f})")
print(f"\nT-interval is {((t_ci_small[1]-t_ci_small[0])/(z_ci_small[1]-z_ci_small[0]) - 1)*100:.1f}% wider!")
print(f"\n✓ This extra width accounts for uncertainty in estimating σ from a small sample")

---
## Part 8: Factors Affecting Interval Width

Three main factors affect the width of a confidence interval:

1. **Sample size (n)** - Larger sample → Narrower interval
2. **Variability (σ or s)** - More variability → Wider interval
3. **Confidence level** - Higher confidence → Wider interval

Let's explore each factor interactively!

### Factor 1: Sample Size Effect

In [ ]:
# Demonstrate sample size effect
confidence_level = 0.95
sample_sizes = [10, 25, 50, 100, 200, 500, 1000]

widths = []
for n in sample_sizes:
    sample = np.random.choice(population, size=n, replace=False)
    sample_std = sample.std(ddof=1)
    se = sample_std / np.sqrt(n)
    t_crit = stats.t.ppf((1 + confidence_level) / 2, df=n-1)
    margin = t_crit * se
    width = 2 * margin
    widths.append(width)

# Plot
plt.figure(figsize=(12, 6))
plt.plot(sample_sizes, widths, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Sample Size (n)')
plt.ylabel('Confidence Interval Width')
plt.title('Effect of Sample Size on 95% CI Width')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Sample Size Effect:")
for n, width in zip(sample_sizes, widths):
    print(f"   n = {n:>4}: CI width = {width:.4f}")

print(f"\n💡 Key insight: Quadrupling the sample size (n=50 → n=200) approximately")
print(f"   halves the CI width ({widths[2]:.3f} → {widths[4]:.3f})")
print(f"   This is because SE ∝ 1/√n")

### Factor 2: Variability Effect

In [ ]:
# Create populations with different variability
n = 100
confidence_level = 0.95
std_devs = [0.5, 1.0, 1.5, 2.0, 2.5]

widths = []
for std in std_devs:
    # Create a population with this std dev
    pop = np.random.normal(7.5, std, 100000)
    pop = np.clip(pop, 1, 10)
    
    # Take a sample
    sample = np.random.choice(pop, size=n, replace=False)
    sample_std = sample.std(ddof=1)
    se = sample_std / np.sqrt(n)
    t_crit = stats.t.ppf((1 + confidence_level) / 2, df=n-1)
    margin = t_crit * se
    width = 2 * margin
    widths.append(width)

# Plot
plt.figure(figsize=(12, 6))
plt.plot(std_devs, widths, 'ro-', linewidth=2, markersize=8)
plt.xlabel('Population Standard Deviation (σ)')
plt.ylabel('Confidence Interval Width')
plt.title(f'Effect of Variability on 95% CI Width (n={n})')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Variability Effect:")
for std, width in zip(std_devs, widths):
    print(f"   σ = {std:.1f}: CI width = {width:.4f}")

print(f"\n💡 Key insight: CI width is directly proportional to standard deviation")
print(f"   More variability in data → Less precise estimates")

### Factor 3: Confidence Level Effect

In [ ]:
# Demonstrate confidence level effect
sample = np.random.choice(population, size=100, replace=False)
sample_mean = sample.mean()
sample_std = sample.std(ddof=1)
n = len(sample)
se = sample_std / np.sqrt(n)

confidence_levels = np.arange(0.80, 1.00, 0.01)
widths = []

for conf in confidence_levels:
    t_crit = stats.t.ppf((1 + conf) / 2, df=n-1)
    margin = t_crit * se
    width = 2 * margin
    widths.append(width)

# Plot
plt.figure(figsize=(12, 6))
plt.plot(confidence_levels * 100, widths, 'g-', linewidth=2)

# Mark common levels
common_levels = [0.90, 0.95, 0.99]
for conf in common_levels:
    t_crit = stats.t.ppf((1 + conf) / 2, df=n-1)
    width = 2 * t_crit * se
    plt.plot(conf * 100, width, 'ro', markersize=10)
    plt.text(conf * 100, width + 0.02, f'{conf*100:.0f}%', ha='center', fontweight='bold')

plt.xlabel('Confidence Level (%)')
plt.ylabel('Confidence Interval Width')
plt.title(f'Effect of Confidence Level on CI Width (n={n})')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Common Confidence Levels:")
for conf in common_levels:
    t_crit = stats.t.ppf((1 + conf) / 2, df=n-1)
    width = 2 * t_crit * se
    print(f"   {conf*100:.0f}% confidence: CI width = {width:.4f}")

print(f"\n💡 Key insight: The confidence-precision trade-off")
print(f"   More confidence → Wider interval (less precise)")
print(f"   Less confidence → Narrower interval (more precise)")

### 🧪 Interactive Exploration

Now you can experiment with all three factors simultaneously:

In [ ]:
# Interactive function to explore all factors
def explore_ci_width(sample_size=100, pop_std=1.8, confidence_level=0.95):
    """
    Explore how different factors affect confidence interval width.
    
    Try different values:
    - sample_size: 10, 50, 100, 500 (larger = narrower CI)
    - pop_std: 0.5, 1.0, 2.0, 3.0 (larger = wider CI)
    - confidence_level: 0.90, 0.95, 0.99 (larger = wider CI)
    """
    # Create population with specified std
    pop = np.random.normal(7.5, pop_std, 100000)
    pop = np.clip(pop, 1, 10)
    
    # Take sample
    sample = np.random.choice(pop, size=sample_size, replace=False)
    sample_mean = sample.mean()
    sample_std = sample.std(ddof=1)
    se = sample_std / np.sqrt(sample_size)
    
    # Calculate CI
    t_crit = stats.t.ppf((1 + confidence_level) / 2, df=sample_size-1)
    margin = t_crit * se
    ci_lower = sample_mean - margin
    ci_upper = sample_mean + margin
    width = ci_upper - ci_lower
    
    # Visualize
    plt.figure(figsize=(12, 5))
    plt.axvspan(ci_lower, ci_upper, alpha=0.3, color='skyblue', label=f'{confidence_level*100:.0f}% CI')
    plt.axvline(sample_mean, color='red', linewidth=2, label='Sample Mean')
    plt.xlabel('Value')
    plt.title(f'Confidence Interval\nn={sample_size}, σ={pop_std:.1f}, confidence={confidence_level*100:.0f}%')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Print results
    print(f"\n📊 Results:")
    print(f"   Sample size: {sample_size}")
    print(f"   Sample mean: {sample_mean:.4f}")
    print(f"   Sample std: {sample_std:.4f}")
    print(f"   Standard error: {se:.4f}")
    print(f"   Critical value (t*): {t_crit:.4f}")
    print(f"   Margin of error: {margin:.4f}")
    print(f"   {confidence_level*100:.0f}% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
    print(f"   CI width: {width:.4f}")
    
    return ci_lower, ci_upper

# Try it out! Modify the parameters:
explore_ci_width(sample_size=100, pop_std=1.8, confidence_level=0.95)

### 🤔 Reflection Questions

1. If you wanted a more precise estimate (narrower CI) but couldn't change the population variability, what could you do?
2. Why might someone choose a 90% CI instead of a 99% CI?
3. By how much would you need to increase the sample size to cut the CI width in half?

---
## Part 9: Real-World Applications

Let's apply confidence intervals to realistic scenarios!

### Example 1: Medical Study - Drug Effectiveness

A pharmaceutical company tests a new blood pressure medication on 50 patients. After treatment, the average reduction in systolic blood pressure is 12.3 mmHg with a standard deviation of 4.2 mmHg.

In [ ]:
# Medical study data
n_patients = 50
mean_reduction = 12.3  # mmHg
std_reduction = 4.2    # mmHg
confidence = 0.95

# Calculate 95% confidence interval
se = std_reduction / np.sqrt(n_patients)
t_crit = stats.t.ppf((1 + confidence) / 2, df=n_patients-1)
margin = t_crit * se
ci_lower = mean_reduction - margin
ci_upper = mean_reduction + margin

print("🏥 Medical Study: Blood Pressure Medication")
print("="*50)
print(f"Sample size: {n_patients} patients")
print(f"Mean reduction: {mean_reduction:.1f} mmHg")
print(f"Standard deviation: {std_reduction:.1f} mmHg")
print(f"\n95% Confidence Interval: [{ci_lower:.2f}, {ci_upper:.2f}] mmHg")
print(f"\n📖 Interpretation:")
print(f"   We are 95% confident that the true mean reduction in blood pressure")
print(f"   for the population is between {ci_lower:.1f} and {ci_upper:.1f} mmHg.")
print(f"\n💊 Clinical significance:")
if ci_lower > 10:
    print(f"   Since the entire interval is above 10 mmHg, we can be confident")
    print(f"   the drug produces a clinically meaningful reduction.")
else:
    print(f"   The interval includes values below 10 mmHg, so we cannot be")
    print(f"   confident about clinically meaningful effects.")

### Example 2: Business Analytics - Customer Satisfaction

An e-commerce company surveys 200 customers about their satisfaction. The average rating is 8.2 out of 10 with a standard deviation of 1.5.

In [ ]:
# Business analytics data
n_customers = 200
mean_rating = 8.2
std_rating = 1.5
confidence = 0.95

# Calculate 95% confidence interval
se = std_rating / np.sqrt(n_customers)
t_crit = stats.t.ppf((1 + confidence) / 2, df=n_customers-1)
margin = t_crit * se
ci_lower = mean_rating - margin
ci_upper = mean_rating + margin

print("🛒 Business Analytics: Customer Satisfaction")
print("="*50)
print(f"Sample size: {n_customers} customers")
print(f"Mean rating: {mean_rating:.1f} / 10")
print(f"Standard deviation: {std_rating:.1f}")
print(f"\n95% Confidence Interval: [{ci_lower:.2f}, {ci_upper:.2f}] / 10")
print(f"\n📖 Interpretation:")
print(f"   We are 95% confident that the true mean satisfaction rating")
print(f"   for all customers is between {ci_lower:.2f} and {ci_upper:.2f}.")
print(f"\n📊 Business decision:")
print(f"   With a lower bound of {ci_lower:.2f}, we can report with high confidence")
print(f"   that customer satisfaction exceeds 8.0, meeting our target.")

# Visualize
plt.figure(figsize=(12, 5))
plt.axvspan(ci_lower, ci_upper, alpha=0.3, color='lightgreen', label='95% CI')
plt.axvline(mean_rating, color='darkgreen', linewidth=3, label='Sample Mean')
plt.axvline(8.0, color='red', linestyle='--', linewidth=2, label='Target (8.0)')
plt.xlabel('Customer Satisfaction Rating')
plt.title('Customer Satisfaction: 95% Confidence Interval')
plt.xlim(7.5, 8.7)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Example 3: Quality Control - Manufacturing

A factory produces bolts with a target diameter of 10.0 mm. A quality inspector measures 30 random bolts and finds a mean diameter of 10.02 mm with standard deviation 0.08 mm.

In [ ]:
# Manufacturing data
n_bolts = 30
mean_diameter = 10.02  # mm
std_diameter = 0.08    # mm
target = 10.00         # mm
tolerance = 0.05       # mm (acceptable deviation)
confidence = 0.99      # Higher confidence for safety-critical application

# Calculate 99% confidence interval
se = std_diameter / np.sqrt(n_bolts)
t_crit = stats.t.ppf((1 + confidence) / 2, df=n_bolts-1)
margin = t_crit * se
ci_lower = mean_diameter - margin
ci_upper = mean_diameter + margin

print("🔧 Quality Control: Bolt Manufacturing")
print("="*50)
print(f"Sample size: {n_bolts} bolts")
print(f"Mean diameter: {mean_diameter:.4f} mm")
print(f"Standard deviation: {std_diameter:.4f} mm")
print(f"Target diameter: {target:.2f} mm ± {tolerance:.2f} mm")
print(f"\n99% Confidence Interval: [{ci_lower:.4f}, {ci_upper:.4f}] mm")
print(f"\n📖 Interpretation:")
print(f"   We are 99% confident that the true mean diameter is between")
print(f"   {ci_lower:.4f} mm and {ci_upper:.4f} mm.")
print(f"\n⚠️  Quality assessment:")
if ci_lower >= target - tolerance and ci_upper <= target + tolerance:
    print(f"   ✓ PASS: The entire confidence interval falls within tolerance.")
    print(f"     The manufacturing process appears to be in control.")
elif abs(mean_diameter - target) <= tolerance:
    print(f"   ⚠️  CAUTION: The mean is within tolerance, but the CI extends outside.")
    print(f"     Some bolts may be out of spec. Consider process adjustments.")
else:
    print(f"   ✗ FAIL: The process appears to be out of control.")
    print(f"     Immediate corrective action required.")

# Visualize
plt.figure(figsize=(12, 6))
plt.axvspan(target - tolerance, target + tolerance, alpha=0.2, color='green', label='Acceptable Range')
plt.axvspan(ci_lower, ci_upper, alpha=0.4, color='blue', label='99% CI')
plt.axvline(mean_diameter, color='darkblue', linewidth=3, label='Sample Mean')
plt.axvline(target, color='red', linestyle='--', linewidth=2, label='Target')
plt.xlabel('Bolt Diameter (mm)')
plt.title('Manufacturing Quality Control: 99% Confidence Interval')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 10: Common Misconceptions and Pitfalls

Let's address the most common mistakes when interpreting confidence intervals.

### Misconception #1: "Probability that the true parameter is in the interval"

❌ **WRONG**: "There's a 95% probability that the true mean is between 7.0 and 7.5"

✅ **CORRECT**: "If we repeated this sampling procedure many times, 95% of the resulting intervals would contain the true mean"

**Why it matters**: The true parameter is fixed (not random), so we can't assign a probability to it. The interval is what varies from sample to sample.

In [ ]:
# Demonstrate: Generate many samples and see which intervals capture the truth
np.random.seed(100)
n_intervals = 50
sample_size = 50
confidence = 0.95

intervals = []
captures = []

for i in range(n_intervals):
    sample = np.random.choice(population, size=sample_size, replace=False)
    sample_mean = sample.mean()
    sample_std = sample.std(ddof=1)
    se = sample_std / np.sqrt(sample_size)
    t_crit = stats.t.ppf((1 + confidence) / 2, df=sample_size-1)
    margin = t_crit * se
    
    ci_lower = sample_mean - margin
    ci_upper = sample_mean + margin
    
    intervals.append((ci_lower, ci_upper, sample_mean))
    captures.append(ci_lower <= TRUE_POPULATION_MEAN <= ci_upper)

capture_rate = sum(captures) / len(captures)

print(f"\n🔍 Repeated Sampling Demonstration ({n_intervals} samples):\n")
print(f"   True population mean: {TRUE_POPULATION_MEAN:.2f} (fixed, never changes)")
print(f"   Confidence level: {confidence*100:.0f}%")
print(f"   Intervals that captured true mean: {sum(captures)} / {n_intervals}")
print(f"   Capture rate: {capture_rate*100:.1f}% (expected: {confidence*100:.0f}%)")
print(f"\n✓ The true mean doesn't have a probability - it's fixed!")
print(f"✓ The intervals vary - 95% of them capture the true value.")

### Misconception #2: "95% of the data falls in the interval"

❌ **WRONG**: "95% of customer satisfaction scores are between 7.0 and 7.5"

✅ **CORRECT**: "We're 95% confident the mean satisfaction score is between 7.0 and 7.5"

**Why it matters**: A confidence interval is about the population **mean**, not individual data points.

In [ ]:
# Demonstrate the difference between CI for mean vs data distribution
sample = np.random.choice(population, size=100, replace=False)
sample_mean = sample.mean()
sample_std = sample.std(ddof=1)
se = sample_std / np.sqrt(len(sample))
t_crit = stats.t.ppf(0.975, df=len(sample)-1)

# 95% CI for the mean
ci_lower = sample_mean - t_crit * se
ci_upper = sample_mean + t_crit * se

# 95% of the data (approximately)
data_lower = sample_mean - 2 * sample_std
data_upper = sample_mean + 2 * sample_std

plt.figure(figsize=(14, 6))

# Plot histogram of data
plt.hist(sample, bins=30, density=True, alpha=0.5, color='lightblue', edgecolor='black', label='Sample Data')

# Mark the mean
plt.axvline(sample_mean, color='red', linewidth=3, label=f'Sample Mean ({sample_mean:.2f})')

# Show 95% CI for the mean
plt.axvspan(ci_lower, ci_upper, alpha=0.3, color='red', label=f'95% CI for Mean [{ci_lower:.2f}, {ci_upper:.2f}]')

# Show where ~95% of data falls
plt.axvline(data_lower, color='blue', linestyle='--', linewidth=2, alpha=0.7)
plt.axvline(data_upper, color='blue', linestyle='--', linewidth=2, alpha=0.7, label=f'~95% of data: [{data_lower:.2f}, {data_upper:.2f}]')

plt.xlabel('Value')
plt.ylabel('Density')
plt.title('Confidence Interval for Mean vs Range of Data')
plt.legend(loc='upper left', fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Key Differences:\n")
print(f"   95% CI for the MEAN: [{ci_lower:.2f}, {ci_upper:.2f}]")
print(f"      → Where we think the population mean is")
print(f"      → Width: {ci_upper - ci_lower:.2f}")
print(f"\n   ~95% of the DATA: [{data_lower:.2f}, {data_upper:.2f}]")
print(f"      → Where most individual values fall")
print(f"      → Width: {data_upper - data_lower:.2f}")
print(f"\n💡 The CI for the mean is much narrower because we're estimating")
print(f"   an average, not predicting individual values!")

### Misconception #3: "We're 95% sure"

The word "confidence" can be misleading. It's not about subjective certainty!

❌ **WRONG**: "I'm 95% sure the true mean is in this interval"

✅ **CORRECT**: "This interval was constructed using a method that captures the true parameter 95% of the time"

**Why it matters**: Confidence is a property of the **method**, not a statement about any particular interval.

### Pitfall: Multiple Comparisons

If you calculate many confidence intervals, some will fail to capture the true value just by chance!

**Example**: Test 20 different hypotheses at 95% confidence → Expect about 1 to be wrong!

In [ ]:
# Demonstrate multiple comparisons problem
n_comparisons = 20
confidence = 0.95

# Create 20 different "populations" (representing different tests)
results = []
for i in range(n_comparisons):
    # Each test has its own population
    true_mean = 7.5 + np.random.normal(0, 0.5)  # Slightly different true means
    test_pop = np.random.normal(true_mean, 1.8, 10000)
    
    # Take a sample and compute CI
    sample = np.random.choice(test_pop, size=50, replace=False)
    sample_mean = sample.mean()
    sample_std = sample.std(ddof=1)
    se = sample_std / np.sqrt(len(sample))
    t_crit = stats.t.ppf((1 + confidence) / 2, df=len(sample)-1)
    margin = t_crit * se
    
    ci_lower = sample_mean - margin
    ci_upper = sample_mean + margin
    captured = ci_lower <= true_mean <= ci_upper
    
    results.append((true_mean, ci_lower, ci_upper, captured))

# Count failures
n_failures = sum(1 for _, _, _, captured in results if not captured)

print(f"\n⚠️  Multiple Comparisons Problem:\n")
print(f"   Number of tests: {n_comparisons}")
print(f"   Confidence level per test: {confidence*100:.0f}%")
print(f"   Expected failures: {n_comparisons * (1 - confidence):.1f}")
print(f"   Actual failures: {n_failures}")
print(f"\n💡 With {n_comparisons} tests at 95% confidence, we expect about")
print(f"   {n_comparisons * (1-confidence):.0f} intervals to miss the true value!")
print(f"\n🔧 Solution: Use corrections like Bonferroni when doing multiple tests.")

---
## Part 11: Using Python Libraries

While it's important to understand the formulas, Python makes confidence interval calculations easy!

In [ ]:
# Method 1: scipy.stats (most direct)
sample = np.random.choice(population, size=100, replace=False)
confidence_level = 0.95

# Calculate confidence interval using scipy
ci = stats.t.interval(confidence=confidence_level,
                      df=len(sample)-1,
                      loc=sample.mean(),
                      scale=stats.sem(sample))  # sem = standard error of mean

print("📦 Method 1: scipy.stats.t.interval()")
print(f"   95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"   Width: {ci[1] - ci[0]:.4f}")

# Method 2: Manual calculation with scipy (more transparent)
sample_mean = sample.mean()
sample_std = sample.std(ddof=1)
n = len(sample)
se = sample_std / np.sqrt(n)
t_crit = stats.t.ppf((1 + confidence_level) / 2, df=n-1)
margin = t_crit * se
ci_manual = (sample_mean - margin, sample_mean + margin)

print(f"\n📦 Method 2: Manual with scipy helpers")
print(f"   95% CI: [{ci_manual[0]:.4f}, {ci_manual[1]:.4f}]")
print(f"   Width: {ci_manual[1] - ci_manual[0]:.4f}")

print(f"\n✓ Both methods give identical results!")
print(f"   Method 1 is convenient, Method 2 is more transparent")

### Creating a Reusable Function

In [ ]:
def calculate_confidence_interval(data, confidence=0.95):
    """
    Calculate confidence interval for the mean.
    
    Parameters:
    - data: array-like, sample data
    - confidence: float, confidence level (default 0.95)
    
    Returns:
    - Dictionary with CI bounds, mean, and other statistics
    """
    data = np.array(data)
    n = len(data)
    mean = data.mean()
    std = data.std(ddof=1)
    se = std / np.sqrt(n)
    
    # Calculate t-critical value
    t_crit = stats.t.ppf((1 + confidence) / 2, df=n-1)
    margin = t_crit * se
    
    ci_lower = mean - margin
    ci_upper = mean + margin
    
    return {
        'mean': mean,
        'std': std,
        'n': n,
        'se': se,
        'confidence': confidence,
        't_critical': t_crit,
        'margin_of_error': margin,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'ci_width': ci_upper - ci_lower
    }

def print_ci_report(ci_result):
    """Print a formatted report of CI results."""
    print(f"\n{'='*60}")
    print(f" Confidence Interval Report")
    print(f"{'='*60}")
    print(f"  Sample size (n):        {ci_result['n']}")
    print(f"  Sample mean:            {ci_result['mean']:.4f}")
    print(f"  Sample std dev:         {ci_result['std']:.4f}")
    print(f"  Standard error:         {ci_result['se']:.4f}")
    print(f"  Confidence level:       {ci_result['confidence']*100:.0f}%")
    print(f"  t-critical value:       {ci_result['t_critical']:.4f}")
    print(f"  Margin of error:        {ci_result['margin_of_error']:.4f}")
    print(f"\n  {ci_result['confidence']*100:.0f}% Confidence Interval: [{ci_result['ci_lower']:.4f}, {ci_result['ci_upper']:.4f}]")
    print(f"  Interval width:         {ci_result['ci_width']:.4f}")
    print(f"{'='*60}\n")

# Test the function
sample = np.random.choice(population, size=75, replace=False)
result = calculate_confidence_interval(sample, confidence=0.95)
print_ci_report(result)

---
## Part 12: Summary and Key Takeaways

Congratulations! You've completed a deep dive into confidence intervals. Let's recap the most important points:

### 🎯 Core Concepts

1. **What is a Confidence Interval?**
   - A range of plausible values for a population parameter
   - Formula: $\text{Point Estimate} \pm (\text{Critical Value} \times \text{Standard Error})$

2. **Correct Interpretation (95% CI)**
   - ✅ "If we repeated sampling many times, 95% of intervals would contain the true parameter"
   - ❌ NOT "95% probability the true parameter is in this interval"
   - ❌ NOT "95% of the data falls in this interval"

3. **Key Components**
   - **Point Estimate**: Usually the sample mean ($\bar{x}$)
   - **Standard Error**: $SE = s / \sqrt{n}$ (measures sampling variability)
   - **Critical Value**: From t-distribution (or z if σ known)
   - **Margin of Error**: Critical value × Standard error

### 📏 Factors Affecting Width

| Factor | Effect | Why |
|--------|--------|-----|
| **Sample size ↑** | Width ↓ | More data → more precision |
| **Variability ↑** | Width ↑ | More spread → less certainty |
| **Confidence level ↑** | Width ↑ | More confidence → cast wider net |

### 🔧 Practical Guidelines

1. **When to use t vs z**
   - Use **t-interval** when population σ unknown (almost always!)
   - Use **z-interval** only when population σ is known

2. **Choosing confidence level**
   - **90%**: Less conservative, narrower intervals
   - **95%**: Standard in most fields
   - **99%**: More conservative, wider intervals (safety-critical applications)

3. **Sample size considerations**
   - Small samples (n < 30): t-distribution is essential
   - Large samples (n ≥ 30): t and z give similar results
   - To halve CI width: Need to quadruple sample size

### ⚠️ Common Pitfalls to Avoid

1. Confusing confidence with probability
2. Thinking CI describes the data spread (it describes the mean)
3. Forgetting about multiple comparisons
4. Using z when you should use t
5. Ignoring the assumptions (normality, independence, random sampling)

### 🧪 Final Challenge: Test Your Understanding

Try these exercises to solidify your knowledge:

In [ ]:
# Exercise 1: Calculate a 90% CI for this sample
exercise_sample_1 = np.array([23, 25, 22, 28, 24, 26, 25, 27, 23, 24])

print("Exercise 1: Calculate a 90% confidence interval")
print(f"Sample: {exercise_sample_1}")
print("\nYour turn: Calculate the 90% CI using the function we created!")

# Uncomment and complete:
# result1 = calculate_confidence_interval(exercise_sample_1, confidence=0.90)
# print_ci_report(result1)

Display the output.

In [ ]:
# Exercise 2: Compare confidence levels
exercise_sample_2 = np.random.normal(100, 15, 50)

print("Exercise 2: Compare different confidence levels")
print(f"Sample size: {len(exercise_sample_2)}")
print(f"Sample mean: {exercise_sample_2.mean():.2f}")
print("\nCalculate 90%, 95%, and 99% CIs and compare their widths!")

# Your turn: Calculate and compare!

Display the output.

In [ ]:
# Exercise 3: Real-world interpretation
# A company tests a new training program on 40 employees
# Productivity scores (out of 100) after training:
productivity_scores = np.random.normal(78, 12, 40)

print("Exercise 3: Real-world application")
print("\nScenario: A company implements a new training program.")
print("The goal is to achieve a mean productivity score of at least 75.")
print(f"\nSample size: {len(productivity_scores)} employees")
print(f"Sample mean: {productivity_scores.mean():.2f}")
print(f"Sample std: {productivity_scores.std(ddof=1):.2f}")
print("\nTask: Calculate a 95% CI and determine if the company can be confident")
print("      they've met their goal of mean score ≥ 75.")

# Your turn: Calculate CI and interpret!

### 📚 Next Steps

Now that you understand confidence intervals, you're ready to explore:

1. **Hypothesis Testing** - Use CIs to make decisions
2. **Confidence Intervals for Proportions** - When data is categorical
3. **Confidence Intervals for Differences** - Comparing two groups
4. **Bootstrap Confidence Intervals** - Non-parametric alternative
5. **Prediction Intervals** - Intervals for individual predictions (not means)

### 🎓 Congratulations!

You now have a solid, intuitive understanding of confidence intervals. Remember:
- Confidence is about the **method**, not any particular interval
- Always interpret CIs in context
- Consider the practical significance, not just statistical significance

Happy analyzing! 📊